# Whisper ASR — 32 video failed (bỏ L24 sport) — tải qua Range + transcribe 2 GPU

Mỗi video được lấy đúng byte-range từ server lot ZIP (không tải cả ZIP, tổng ~7.3GB) → `faster-whisper large-v3` → `.jsonl` → xóa video ngay.

**Chạy: Save & Run All** để output được snapshot; tải về bằng `scripts/download_kaggle_transcripts.py`.


In [ ]:
# CELL 1 — Install
!pip install -q faster-whisper

In [ ]:
# CELL 2 — Load model (tự phát hiện số GPU), cache về /tmp
import os, json, time, random, shutil, queue, threading, requests
from pathlib import Path

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_HUB_CACHE"] = "/tmp/hf_cache"

from faster_whisper import WhisperModel

try:
    import torch
    NUM_GPUS = max(1, torch.cuda.device_count())
except Exception:
    NUM_GPUS = 1

print(f"Phát hiện {NUM_GPUS} GPU — tải {NUM_GPUS} model large-v3 ...")
models = [WhisperModel("large-v3", device=f"cuda:{i}", compute_type="float16",
                       download_root="/tmp/whisper_models") for i in range(NUM_GPUS)]
model_q = queue.Queue()
for m in models:
    model_q.put(m)
print(f"Loaded {len(models)} model(s).")

for d in ["/root/.cache/pip", "/kaggle/working/.cache"]:
    if Path(d).exists():
        shutil.rmtree(d, ignore_errors=True)
        print(f"  Freed {d}")

In [ ]:
# CELL 3 — Index video (data_offset/size) + fetch qua Range (resume, retry, validate MP4)
VIDEO_FILES = {
"L26_V298": {
"data_offset": 6769867441,
"size": 71491243,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L26_c.zip"
},
"L26_V313": {
"data_offset": 873634025,
"size": 70023261,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L26_d.zip"
},
"L26_V337": {
"data_offset": 2537434867,
"size": 65079290,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L26_d.zip"
},
"L26_V381": {
"data_offset": 5485640888,
"size": 66890320,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L26_d.zip"
},
"L26_V432": {
"data_offset": 2109648328,
"size": 71627060,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L26_e.zip"
},
"L26_V491": {
"data_offset": 6272083191,
"size": 76617879,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L26_e.zip"
},
"L28_V001": {
"data_offset": 76,
"size": 341197784,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V002": {
"data_offset": 341197936,
"size": 289731546,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V003": {
"data_offset": 630929558,
"size": 291297063,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V004": {
"data_offset": 922226697,
"size": 308206598,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V005": {
"data_offset": 1230433371,
"size": 284481841,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V006": {
"data_offset": 1514915288,
"size": 309638121,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V007": {
"data_offset": 1824553485,
"size": 304306957,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V008": {
"data_offset": 2128860518,
"size": 291407261,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V009": {
"data_offset": 2420267855,
"size": 275803635,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V010": {
"data_offset": 2696071566,
"size": 272706129,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V011": {
"data_offset": 2968777771,
"size": 308922871,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V012": {
"data_offset": 3277700718,
"size": 308242550,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V013": {
"data_offset": 3585943344,
"size": 312113282,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V014": {
"data_offset": 3898056702,
"size": 290015371,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V015": {
"data_offset": 4188072149,
"size": 301707415,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V016": {
"data_offset": 4489779640,
"size": 329526885,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V017": {
"data_offset": 4819306601,
"size": 331339590,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V018": {
"data_offset": 5150646267,
"size": 308152428,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V019": {
"data_offset": 5458798771,
"size": 304317090,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V020": {
"data_offset": 5763115937,
"size": 314771069,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V021": {
"data_offset": 6077887082,
"size": 290897227,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V022": {
"data_offset": 6368784385,
"size": 314222158,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V023": {
"data_offset": 6683006619,
"size": 281193147,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L28_V024": {
"data_offset": 6964199842,
"size": 310283365,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L28_a.zip"
},
"L30_V029": {
"data_offset": 1141875514,
"size": 59435109,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L30_a.zip"
},
"L30_V039": {
"data_offset": 1648061834,
"size": 48720009,
"zip_url": "https://aic-data.ledo.io.vn/Videos_L30_a.zip"
}
}
VIDEO_ORDER = ["L28_V001", "L28_V002", "L28_V003", "L28_V004", "L28_V005", "L28_V006", "L28_V007", "L28_V008", "L28_V009", "L28_V010", "L28_V011", "L28_V012", "L28_V013", "L28_V014", "L28_V015", "L28_V016", "L28_V017", "L28_V018", "L28_V019", "L28_V020", "L28_V021", "L28_V022", "L28_V023", "L28_V024", "L26_V298", "L26_V313", "L26_V337", "L26_V381", "L26_V432", "L26_V491", "L30_V029", "L30_V039"]

OUT_DIR = Path("/kaggle/tmp/transcripts")
OUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR = Path("/kaggle/tmp/videos")
VIDEO_DIR.mkdir(exist_ok=True)

def fetch_video(vid, info, dest, max_retries=8, max_validate=3):
    """Tải đúng range của video. Resume khi đứt; fallback cắt luồng khi server bỏ Range;
    validate 'ftyp' (MP4) — nếu sai byte (index lệch) thì tải lại từ đầu, tối đa max_validate lần."""
    url, start, size = info["zip_url"], info["data_offset"], info["size"]
    for val in range(max_validate):
        for attempt in range(max_retries):
            try:
                got = dest.stat().st_size if dest.exists() else 0
                if got >= size:
                    break
                headers = {"Range": f"bytes={start+got}-{start+size-1}"}
                r = requests.get(url, stream=True, headers=headers,
                                 allow_redirects=True, timeout=(10, 600))
                if r.status_code == 206:
                    with open(dest, "ab" if got else "wb") as f:
                        for chunk in r.iter_content(8 * 1024 * 1024):
                            f.write(chunk)
                elif r.status_code == 200:
                    skip, remain = start + got, size - got
                    with open(dest, "ab" if got else "wb") as f:
                        for chunk in r.iter_content(8 * 1024 * 1024):
                            if skip >= len(chunk):
                                skip -= len(chunk)
                                continue
                            chunk = chunk[skip:]
                            skip = 0
                            if len(chunk) > remain:
                                chunk = chunk[:remain]
                            f.write(chunk)
                            remain -= len(chunk)
                            if remain <= 0:
                                break
                else:
                    raise RuntimeError(f"HTTP {r.status_code}")
                if dest.stat().st_size >= size:
                    break
            except Exception as e:
                print(f"  [{vid}] retry {attempt+1}/{max_retries}: {type(e).__name__}: {str(e)[:100]}")
                time.sleep(min(2 ** attempt, 60) + random.uniform(0, 2))
        if dest.stat().st_size >= size:
            head = dest.read_bytes()[:16]
            if b"ftyp" in head:
                return dest
            print(f"  [{vid}] byte không phải MP4 (validate {val+1}/{max_validate}) — tải lại")
            dest.unlink(missing_ok=True)
        else:
            raise RuntimeError(f"fetch failed after {max_retries} retries")
    raise RuntimeError(f"MP4 validate failed after {max_validate} attempts (index lệch?)")

In [ ]:
# CELL 4 — Pipeline: tải song song 3 luồng + transcribe song song trên từng GPU → jsonl → xóa video
DOWNLOAD_AHEAD = 3      # luồng tải nền (video nhỏ nên 3 vẫn rất nhẹ disk)
BEAM_SIZE = 5
print_lock = threading.Lock()
stats = {"ok": 0, "fail": 0, "dlfail": 0, "skip": 0}
failed_videos = []

def _dl():
    while True:
        vid = in_q.get()
        if vid is None:
            in_q.task_done()
            break
        dest = VIDEO_DIR / f"{vid}.mp4"
        try:
            fetch_video(vid, VIDEO_FILES[vid], dest)
            out_q.put((vid, dest))
        except Exception as e:
            with print_lock:
                print(f"  FAIL tải {vid}: {e}")
            out_q.put((vid, None))
        in_q.task_done()

def _tr():
    while True:
        job = tr_q.get()
        if job is None:
            tr_q.task_done()
            break
        vid, mp4_path = job
        try:
            out_path = OUT_DIR / f"{vid}.jsonl"
            if out_path.exists():
                with print_lock:
                    print(f"  SKIP {vid}")
                    stats["skip"] += 1
                mp4_path.unlink(missing_ok=True)
                tr_q.task_done()
                continue

            m = model_q.get()
            try:
                t1 = time.perf_counter()
                segs = []
                for attempt in range(2):
                    segs_iter, info = m.transcribe(str(mp4_path), language="vi",
                                                   beam_size=BEAM_SIZE, word_timestamps=True)
                    segs = list(segs_iter)
                    if segs:
                        break
                    print(f"  [WARN {vid}] transcript rỗng — thử lại lần {attempt+2}/2")
                trans_time = time.perf_counter() - t1
            finally:
                model_q.put(m)
            mp4_path.unlink(missing_ok=True)   # xóa video ngay

            if not segs:
                with print_lock:
                    print(f"  FAIL {vid} — empty")
                    stats["fail"] += 1
                    failed_videos.append(vid)
                tr_q.task_done()
                continue

            with open(out_path, "w", encoding="utf-8") as f:
                for s in segs:
                    f.write(json.dumps({
                        "start_time_ms": round(s.start * 1000),
                        "end_time_ms": round(s.end * 1000),
                        "text": s.text.strip(),
                    }, ensure_ascii=False) + "\n")

            dur = segs[-1].end
            with print_lock:
                print(f"  OK   {vid} — {len(segs)} segs | {dur:.0f}s | {trans_time:.1f}s ({dur/max(trans_time,0.1):.1f}x)")
                stats["ok"] += 1
        except Exception as e:
            mp4_path.unlink(missing_ok=True)
            with print_lock:
                print(f"  FAIL {vid} — transcribe lỗi: {type(e).__name__}: {str(e)[:100]}")
                stats["fail"] += 1
                failed_videos.append(vid)
        tr_q.task_done()

in_q = queue.Queue()
out_q = queue.Queue(maxsize=DOWNLOAD_AHEAD)
tr_q = queue.Queue(maxsize=NUM_GPUS * 2)
for vid in VIDEO_ORDER:
    in_q.put(vid)

run_start = time.perf_counter()
dl_threads = [threading.Thread(target=_dl, daemon=True) for _ in range(DOWNLOAD_AHEAD)]
tr_threads = [threading.Thread(target=_tr, daemon=True) for _ in range(NUM_GPUS)]
for t in dl_threads + tr_threads:
    t.start()

for _ in range(len(VIDEO_ORDER)):
    vid, mp4_path = out_q.get()
    out_q.task_done()
    if mp4_path is None:
        with print_lock:
            stats["dlfail"] += 1
            failed_videos.append(vid)
        continue
    tr_q.put((vid, mp4_path))

for t in dl_threads:
    in_q.put(None)
for t in tr_threads:
    tr_q.put(None)
for t in dl_threads + tr_threads:
    t.join()

print(f"\n{'='*50}")
print(f"ALL DONE — OK={stats['ok']} SKIP={stats['skip']} FAIL(transcribe)={stats['fail']} FAIL(download)={stats['dlfail']} in {(time.perf_counter()-run_start)/60:.1f} min")
if failed_videos:
    print("Video cần chạy lại:", " ".join(failed_videos))
print(f"Transcripts: {OUT_DIR} ({len(list(OUT_DIR.glob('*.jsonl')))} JSONL files)")

In [ ]:
# CELL 5 — Copy sang /kaggle/working (Kaggle snapshot output khi Save & Run All) + zip tiện tải tay
import subprocess

WORK_OUT = Path("/kaggle/working/transcripts")
WORK_OUT.mkdir(parents=True, exist_ok=True)
for p in OUT_DIR.glob("*.jsonl"):
    shutil.copy2(p, WORK_OUT / p.name)
print(f"Copied {len(list(WORK_OUT.glob('*.jsonl')))} jsonl → {WORK_OUT}")

subprocess.run(["zip", "-r", "transcripts_whisper_32videos.zip", "transcripts"],
               cwd="/kaggle/working", check=False)

from IPython.display import FileLink
FileLink("/kaggle/working/transcripts_whisper_32videos.zip")